In [1]:
# 1. 라이브러리 불러오기
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 2. 간단한 학습용 문장쌍 정의
eng_sentences = ['hello', 'how are you', 'good morning', 'thank you', 'nice to meet you',
                 'see you later', 'good night', 'i am fine', 'have a nice day', 'excuse me']
kor_sentences = ['안녕', '어떻게 지내', '좋은 아침', '고마워', '반가워',
                 '나중에 봐', '잘 자', '괜찮아', '좋은 하루 보내', '실례합니다']

# 3. 디코더 입력/출력용으로 <start>, <end> 토큰 추가
kor_sentences_input = ['<start> ' + s for s in kor_sentences]
kor_sentences_target = [s + ' <end>' for s in kor_sentences]

# 4. 단어 수준 Tokenizer로 인덱스화
eng_tokenizer = Tokenizer(filters='!"#$%&()*+,-./:;=?@[\\]^_`{|}~\t\n<>')
eng_tokenizer.fit_on_texts(eng_sentences)
eng_sequences = eng_tokenizer.texts_to_sequences(eng_sentences)

kor_tokenizer = Tokenizer(filters='!"#$%&()*+,-./:;=?@[\\]^_`{|}~\t\n')  # <, >는 보존
kor_tokenizer.fit_on_texts(kor_sentences_input + kor_sentences_target)
kor_input_seq  = kor_tokenizer.texts_to_sequences(kor_sentences_input)
kor_target_seq = kor_tokenizer.texts_to_sequences(kor_sentences_target)

# 단어 집합 크기 정의 (+1은 패딩용 0번 포함)
eng_vocab_size = len(eng_tokenizer.word_index) + 1
kor_vocab_size = len(kor_tokenizer.word_index) + 1

# 시퀀스 최대 길이 계산
max_encoder_seq_length = max(len(seq) for seq in eng_sequences)
max_decoder_seq_length = max(len(seq) for seq in kor_input_seq)

# 시퀀스 패딩 (길이 맞춤)
encoder_input_data = pad_sequences(eng_sequences, maxlen=max_encoder_seq_length, padding='post')
decoder_input_data = pad_sequences(kor_input_seq, maxlen=max_decoder_seq_length, padding='post')
decoder_target_data = pad_sequences(kor_target_seq, maxlen=max_decoder_seq_length, padding='post')

# 5. 인코더 정의
latent_dim = 256  # LSTM 출력 차원, Embedding 차원

encoder_inputs = Input(shape=(None,))  # 단어 시퀀스 입력
enc_emb = Embedding(eng_vocab_size, latent_dim)(encoder_inputs)
_, state_h, state_c = LSTM(latent_dim, return_state=True)(enc_emb)  # 마지막 상태만 반환
encoder_states = [state_h, state_c]  # 디코더 초기 상태로 전달

# 6. 디코더 정의
decoder_inputs = Input(shape=(None,))
dec_emb_layer = Embedding(kor_vocab_size, latent_dim)
dec_emb = dec_emb_layer(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense = Dense(kor_vocab_size, activation='softmax')  # 단어 분류용 출력층
decoder_outputs = decoder_dense(decoder_outputs)

# 7. 전체 모델 구성 및 학습
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

model.fit([encoder_input_data, decoder_input_data], decoder_target_data, batch_size=2, epochs=300, verbose=0)

# 8. 추론용 인코더 모델 구성
encoder_model = Model(encoder_inputs, encoder_states)

# 9. 추론용 디코더 모델 구성
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

# 디코더 입력과 상태를 기반으로 다음 단어 예측
dec_emb2 = dec_emb_layer(decoder_inputs)  # 학습된 임베딩 레이어 재사용
decoder_outputs2, state_h2, state_c2 = decoder_lstm(dec_emb2, initial_state=decoder_states_inputs)
decoder_states2 = [state_h2, state_c2]
decoder_outputs2 = decoder_dense(decoder_outputs2)

decoder_model = Model([decoder_inputs] + decoder_states_inputs,
                      [decoder_outputs2] + decoder_states2)

# 10. 인덱스를 단어로 바꾸는 매핑
index2word_kor = {i: w for w, i in kor_tokenizer.word_index.items()}
index2word_kor[0] = '<pad>'  # 패딩

# 11. 번역 함수 정의
def decode_sequence(input_seq):
    # 인코더로부터 초기 상태(context vector) 추출
    states_value = encoder_model.predict(input_seq)

    # 디코더 시작은 <start> 토큰
    start_token_idx = kor_tokenizer.word_index['<start>']
    target_seq = np.array([[start_token_idx]])

    decoded_sentence = ''
    stop_condition = False

    while not stop_condition:
        # 디코더 예측
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)

        # 예측된 단어 선택
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = index2word_kor[sampled_token_index]

        if sampled_word == '<end>' or len(decoded_sentence.split()) >= max_decoder_seq_length:
            stop_condition = True
        elif sampled_word not in ['<start>', '<pad>']:
            decoded_sentence += sampled_word + ' '

        # 다음 입력을 현재 예측된 단어로 설정
        target_seq = np.array([[sampled_token_index]])
        states_value = [h, c]

    return decoded_sentence.strip()

# 12. 사용자 번역 테스트 함수
def translate(input_text):
    seq = eng_tokenizer.texts_to_sequences([input_text])
    seq_padded = pad_sequences(seq, maxlen=max_encoder_seq_length, padding='post')
    result = decode_sequence(seq_padded)
    print(f"\n▶ 입력: {input_text}")
    print(f"👉 번역: {result}")

# 13. 예시 실행
translate("nice to meet you")
translate("thank you")
translate("have a nice day")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step

▶ 입력: nice to meet you
👉 번역: 반가워
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step

▶ 입력: thank you
👉 번역: 고마워
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step

▶ 입력: have a nice day
👉 번역: 좋은 하루 보내
